# 01 — Data exploration with pandas (Phase 1)

**Goal:** understand what the files look like *before* doing any ML.

Everything here is plain pandas:
1. Load a file with `pd.read_csv`.
2. Look at it with `.head()`, `.shape`, `.info()`, `.value_counts()`, `.str.len()`, `.nunique()` ...
3. Delete it with `del` before loading the next one (the laptop has ~8 GB RAM, so we keep one big table in memory at a time).

Nothing in `dataset/` is modified.

In [1]:
import gc
import io

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

TRAIN_DIR = "../dataset/train/"
TEST_DIR = "../dataset/test/"

## 1. Why `sep="\t"`?

TSV = **Tab**-Separated Values. Each column is separated by an invisible TAB character.
Addresses contain commas (`"85 Wayne Avenue, Ticonderoga, NY"`), so if we split on commas,
one address would be broken into several fake columns.

In [2]:
one_line = "entity_id\tbusiness_name\tbusiness_address\tcountry\nS1-1\tMaure Williams Inc\t85 Wayne Avenue, Ticonderoga, NY\tUS\n"

print("Split on TAB (correct):")
display(pd.read_csv(io.StringIO(one_line), sep="\t"))

print("Split on COMMA (wrong) - the address is chopped up and the columns are garbage:")
display(pd.read_csv(io.StringIO(one_line), sep=","))

Split on TAB (correct):


,entity_id,business_name,business_address,country
0,S1-1,Maure Williams Inc,"85 Wayne Avenue, Ticonderoga, NY",US


Split on COMMA (wrong) - the address is chopped up and the columns are garbage:


,,entity_id\tbusiness_name\tbusiness_address\tcountry
S1-1\tMaure Williams Inc\t85 Wayne Avenue,Ticonderoga,NY\tUS


## 2. Why `keep_default_na=False`?

By default pandas turns empty cells **and** words like `"null"`, `"NA"`, `"None"` into `NaN` (missing).
Our addresses really contain the word `"null"` as text, and a business could be called "NA Traders".
With `keep_default_na=False`, empty cells become the empty string `""` and every word is kept as written.
So in this notebook **"missing" means `== ""`**.

## 3. Train Source 1 — step by step

In [3]:
s1_train = pd.read_csv(TRAIN_DIR + "train_source1.tsv", sep="\t", keep_default_na=False)
s1_train.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India


In [4]:
print("shape (rows, columns):", s1_train.shape)
s1_train.info(memory_usage="deep")

shape (rows, columns): (2206821, 4)
<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 2206820
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 259.3 MB


### Missing values

`s1_train == ""` gives a True/False table; `.sum()` counts the Trues per column
(True counts as 1).

In [5]:
(s1_train == "").sum()

entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

### Duplicate IDs

Every `entity_id` must be unique, otherwise our output files would be ambiguous.
`.duplicated()` marks every repeat of a value that already appeared above.

In [6]:
print("duplicate entity_ids:", s1_train["entity_id"].duplicated().sum())
print("fully duplicated rows:", s1_train.duplicated().sum())

duplicate entity_ids: 0


fully duplicated rows: 0


### Countries

`value_counts()` counts how many rows have each value.

In [7]:
s1_train["country"].value_counts()

country
US       1323633
India     883188
Name: count, dtype: int64

### String lengths

`.str.len()` gives the number of characters in each text; `.describe()` summarises those numbers.

In [8]:
lengths = pd.DataFrame({
    "name_length": s1_train["business_name"].str.len(),
    "address_length": s1_train["business_address"].str.len(),
})
lengths.describe().round(1)

,name_length,address_length
count,2206821.0,2206821.0
mean,24.0,52.1
std,7.7,25.3
min,3.0,11.0
25%,18.0,33.0
50%,24.0,41.0
75%,30.0,70.0
max,105.0,256.0


In [9]:
print("Shortest names:")
display(s1_train.loc[lengths["name_length"].nsmallest(5).index, ["entity_id", "business_name"]])
print("Longest names:")
display(s1_train.loc[lengths["name_length"].nlargest(5).index, ["entity_id", "business_name"]])

Shortest names:


,entity_id,business_name
8682,S1-412318813,Raa
9999,S1-831940557,Beo
14027,S1-350751911,Dao
14345,S1-797748666,Bia
26303,S1-813056489,Lia


Longest names:


,entity_id,business_name
1774111,S1-590655896,164/B-43 Vinayaka.Plaza 2Nd Floor 24Th Cross Ro Ad 6Th Block Jayanagar Banga...
1783885,S1-821281223,First Floor Survey No. 134/2/2/1B Bld/B Mp/1 Patil Heritage Shivajinagar Tec...
2163005,S1-64575188,Murlidhar Apartmentajit Park Society Opp Krushi Farm Ghod-Dod Road Calibrati...
908475,S1-643919451,303&304 Dabour Building C-I Local Shopping Centre Iind Floor Janak Puri Elec...
355241,S1-872576541,Door No 8-3-680Prabhava Mansion Navodaya Colony Yekkareddy Uda Creative Priv...


### Unique names and addresses

`.nunique()` counts distinct values. If there are fewer unique names than rows,
several records share the same name.

In [10]:
print(f"rows:              {len(s1_train):,}")
print(f"unique names:      {s1_train['business_name'].nunique():,}")
print(f"unique addresses:  {s1_train['business_address'].nunique():,}")

print("\nMost repeated names:")
s1_train["business_name"].value_counts().head(10)

rows:              2,206,821


unique names:      1,539,229


unique addresses:  2,130,606

Most repeated names:


business_name
Primary Care Group         253
Ear Nose & Throat Group    251
Pediatric Group            222
Womens Health Group        220
Physical Therapy Group     218
Pediatric Dental Group     216
Behavioral Health Group    215
Chiropractic Group         209
Orthopedic Group           208
Eye Group                  207
Name: count, dtype: int64

We keep `s1_train` in memory: we need it again in section 6 for the examples.

## 4. All six source files — the same checks in one table

Same pandas calls as above, done in a loop. Each file is loaded, summarised into one row, and deleted.

In [11]:
files = {
    "train_source1": TRAIN_DIR + "train_source1.tsv",
    "train_source2": TRAIN_DIR + "train_source2.tsv",
    "train_source3": TRAIN_DIR + "train_source3.tsv",
    "test_source1": TEST_DIR + "test_source1.tsv",
    "test_source2": TEST_DIR + "test_source2.tsv",
    "test_source3": TEST_DIR + "test_source3.tsv",
}

summary_rows = []
country_counts = {}

for name, path in files.items():
    df = pd.read_csv(path, sep="\t", keep_default_na=False)
    summary_rows.append({
        "file": name,
        "rows": len(df),
        "memory_MB": round(df.memory_usage(deep=True).sum() / 1e6),
        "duplicate_ids": df["entity_id"].duplicated().sum(),
        "empty_names": (df["business_name"] == "").sum(),
        "empty_addresses": (df["business_address"] == "").sum(),
        "median_name_len": df["business_name"].str.len().median(),
        "median_address_len": df["business_address"].str.len().median(),
        "unique_names": df["business_name"].nunique(),
        "unique_addresses": df["business_address"].nunique(),
    })
    country_counts[name] = df["country"].value_counts()
    print("done:", name, df.shape)
    del df
    gc.collect()

summary = pd.DataFrame(summary_rows).set_index("file")
summary

done: train_source1 (2206821, 4)


done: train_source2 (5034616, 4)


done: train_source3 (5285603, 4)


done: test_source1 (1732544, 4)


done: test_source2 (4887273, 4)


done: test_source3 (5082316, 4)


,rows,memory_MB,duplicate_ids,empty_names,empty_addresses,median_name_len,median_address_len,unique_names,unique_addresses
file,,,,,,,,,
train_source1,2206821,272,0,0,0,24.0,41.0,1539229,2130606
train_source2,5034616,630,0,0,168967,25.0,37.0,4402009,4337262
train_source3,5285603,652,0,0,175916,25.0,42.0,4651609,4632765
test_source1,1732544,224,0,0,0,24.0,50.0,1238867,1677483
test_source2,4887273,646,0,0,129408,25.0,43.0,4311041,4224784
test_source3,5082316,648,0,0,136098,25.0,43.0,4521929,4456436


In [12]:
# one column per file, one row per country (0 = that country does not appear in the file)
pd.DataFrame(country_counts).fillna(0).astype(int)

,train_source1,train_source2,train_source3,test_source1,test_source2,test_source3
country,,,,,,
France,0,0,0,259452,703378,731615
India,883188,2017799,2115547,809986,2312565,2405000
US,1323633,3016817,3170056,663106,1871330,1945701


### What to notice
- **Test has France**, train has only US and India. Our code must treat `country` as an open-ended text field.
- Source 1 addresses are never empty; Source 2 / 3 sometimes are.
- Unique names are fewer than rows → different records share the same name
  (branches, or different businesses with the same name). Name alone is not enough.

## 5. Ground truth (labels) — only exists for train

`train_ground_truth.tsv` has **one row per Source 1 entity**.
`matched_entity_ids` is a comma-separated list, or empty when the business has no match.

In [13]:
gt = pd.read_csv(TRAIN_DIR + "train_ground_truth.tsv", sep="\t", keep_default_na=False)
print("shape:", gt.shape)
gt.head()

shape: (2206821, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


### From one row per S1 to one row per pair

A comma-separated string is hard to count. Two pandas steps fix that:

1. `.str.split(",")` turns `"S2-5,S3-9"` into the list `["S2-5", "S3-9"]`.
2. `.explode()` gives every list element its own row.

```
S1-1   "S2-5,S3-9"      ->   S1-1  S2-5
                              S1-1  S3-9
```

In [14]:
pairs = gt[gt["matched_entity_ids"] != ""].copy()
pairs["match_id"] = pairs["matched_entity_ids"].str.split(",")
pairs = pairs.explode("match_id")[["source1_entity_id", "match_id"]].reset_index(drop=True)
pairs["match_source"] = pairs["match_id"].str[:2]   # "S2" or "S3"

print(f"true (S1, match) pairs:              {len(pairs):,}")
print(f"distinct matched S2/S3 ids:          {pairs['match_id'].nunique():,}")
print(f"S2/S3 ids matched to more than 1 S1: {pairs['match_id'].duplicated().sum():,}")
pairs.head()

true (S1, match) pairs:              7,638,365


distinct matched S2/S3 ids:          7,638,365


S2/S3 ids matched to more than 1 S1: 0


,source1_entity_id,match_id,match_source
0,S1-965667,S2-681193310,S2
1,S1-965667,S2-743505751,S2
2,S1-965667,S3-775321672,S3
3,S1-965667,S3-11291185,S3
4,S1-965667,S3-860443364,S3


### How many matches does one S1 entity have?

`groupby(...).size()` counts rows per S1. S1 entities with no match have no rows in `pairs`,
so `.reindex(..., fill_value=0)` adds them back with a count of 0.

In [15]:
all_s1_ids = gt["source1_entity_id"]

total = pairs.groupby("source1_entity_id").size().reindex(all_s1_ids, fill_value=0)
from_s2 = pairs[pairs["match_source"] == "S2"].groupby("source1_entity_id").size().reindex(all_s1_ids, fill_value=0)
from_s3 = pairs[pairs["match_source"] == "S3"].groupby("source1_entity_id").size().reindex(all_s1_ids, fill_value=0)

distribution = pd.DataFrame({
    "total matches": total.value_counts(),
    "from S2": from_s2.value_counts(),
    "from S3": from_s3.value_counts(),
}).fillna(0).astype(int).sort_index()
distribution.index.name = "number of matches"

print(f"S1 entities with NO match: {(total == 0).sum():,} ({(total == 0).mean():.1%})")
distribution

S1 entities with NO match: 123,247 (5.6%)


,total matches,from S2,from S3
number of matches,,,
0,123247,287745,266276
1,119157,789108,716417
2,375212,652779,668375
3,530841,333957,372443
4,484115,119078,145116
5,321957,24154,35378
6,164868,0,2816
7,63968,0,0
8,18680,0,0


## 6. How the records connect — real examples side by side

1. Pick a few S1 ids.
2. Look up their matched ids in `pairs`.
3. Load Source 2 and Source 3, keep only those rows with `.isin(...)`, and delete the rest.
4. Stack the S1 row and its matches with `pd.concat` to compare them.

In [16]:
example_ids = (
    gt[gt["matched_entity_ids"] != ""].sample(3, random_state=42)["source1_entity_id"].tolist()
    + ["S1-55344266"]                                                                   # Tamil-script example
    + gt[gt["matched_entity_ids"] == ""].sample(1, random_state=42)["source1_entity_id"].tolist()  # no match
)
wanted_match_ids = pairs.loc[pairs["source1_entity_id"].isin(example_ids), "match_id"]

s2_train = pd.read_csv(TRAIN_DIR + "train_source2.tsv", sep="\t", keep_default_na=False)
s2_examples = s2_train[s2_train["entity_id"].isin(wanted_match_ids)]
del s2_train
gc.collect()

s3_train = pd.read_csv(TRAIN_DIR + "train_source3.tsv", sep="\t", keep_default_na=False)
s3_examples = s3_train[s3_train["entity_id"].isin(wanted_match_ids)]
del s3_train
gc.collect()

match_examples = pd.concat([s2_examples, s3_examples])
match_examples

,entity_id,business_name,business_address,country
1886844,S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
1976169,S2-670035601,3520 MAIN ROAD REALTY INC,"SIMPSON LANE, SOMERSET, TX",US
1989598,S2-523219173,MS [Consultancy],"SHYMALA APPTS 1-1ST FLOOR FLAT NO. 4 OPP RATNA HOSPITAL SB ROAD IN HAVELI, P...",India
3650555,S2-407524980,DAVIS FAMILY OFFICE,"OLIVE CIR, LEBANON, TN",US
4130950,S2-249013014,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
4841136,S2-12029274,Davis Family Offie,"88 OLIVE CIR, LEBANON, TN",US
805792,S3-384364074,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு",India
1348219,S3-478195123,Raj Investments எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN",India
2581005,S3-970714334,Davis Family (Office),"88 Olive Cir, Lebanon, Tennessee",US
2630551,S3-716445195,M5 Consultancy Corp,"Pune, MH, Pune, Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Roa...",India


In [17]:
for s1_id in example_ids:
    ids = pairs.loc[pairs["source1_entity_id"] == s1_id, "match_id"]
    print(f"\n########## {s1_id}  ->  {len(ids)} match(es): {', '.join(ids) if len(ids) else '(none)'}")
    side_by_side = pd.concat([
        s1_train[s1_train["entity_id"] == s1_id],
        match_examples[match_examples["entity_id"].isin(ids)],
    ])
    display(side_by_side.reset_index(drop=True))


########## S1-777210964  ->  4 match(es): S2-12029274, S2-407524980, S3-970714334, S3-712922821


,entity_id,business_name,business_address,country
0,S1-777210964,Davis Family Office,"88 Olive Circle, Lebanon, TN",US
1,S2-407524980,DAVIS FAMILY OFFICE,"OLIVE CIR, LEBANON, TN",US
2,S2-12029274,Davis Family Offie,"88 OLIVE CIR, LEBANON, TN",US
3,S3-970714334,Davis Family (Office),"88 Olive Cir, Lebanon, Tennessee",US
4,S3-712922821,Davis Family,"Lebanon, Tennessee, 88 Olive Cir",US



########## S1-190133982  ->  3 match(es): S2-523219173, S3-716445195, S3-986804760


,entity_id,business_name,business_address,country
0,S1-190133982,MS Consultancy Corp,"Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pun...",India
1,S2-523219173,MS [Consultancy],"SHYMALA APPTS 1-1ST FLOOR FLAT NO. 4 OPP RATNA HOSPITAL SB ROAD IN HAVELI, P...",India
2,S3-716445195,M5 Consultancy Corp,"Pune, MH, Pune, Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Roa...",India
3,S3-986804760,MS Corp Services,"Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pun...",India



########## S1-970131671  ->  2 match(es): S2-670035601, S3-392756625


,entity_id,business_name,business_address,country
0,S1-970131671,3520 Main Road Realty Inc,"TX, 158 Simpson Lane, Somerset",US
1,S2-670035601,3520 MAIN ROAD REALTY INC,"SIMPSON LANE, SOMERSET, TX",US
2,S3-392756625,3520 MAIN ROAD Realty INC,"158 Simpson Lane, Somerset, Texas",US



########## S1-55344266  ->  4 match(es): S2-249013014, S2-197070651, S3-478195123, S3-384364074


,entity_id,business_name,business_address,country
0,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",India
1,S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
2,S2-249013014,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
3,S3-384364074,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு",India
4,S3-478195123,Raj Investments எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN",India



########## S1-303396463  ->  0 match(es): (none)


,entity_id,business_name,business_address,country
0,S1-303396463,Indo Super Impex Private Limited,"242 - Ground Floor, Mumbai, Geeta Nagar, Dr Homi Bhabha Road, Navy Nagar, Co...",India


### What to notice in these examples
- **Typos:** "Office" vs "Offie", "MS" vs "M5".
- **Different scripts:** the same Indian business written in English and in Tamil script.
- **Missing parts:** house numbers or whole addresses dropped in some Source 2/3 records.
- **Reordered addresses:** "Lebanon, Tennessee, 88 Olive Cir" vs "88 Olive Circle, Lebanon, TN".
- **Sometimes the name changes a lot** and only the address connects the records.
- **Some S1 entities have no match at all**, so our system must be allowed to answer "none".